# NumPy Mastery for Machine Learning
Comprehensive guide to NumPy operations essential for ML.

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Array Creation and Shapes

In [ ]:
# Various ways to create arrays
a = np.array([1, 2, 3])
b = np.zeros((3, 4))
c = np.ones((2, 3, 4))
d = np.arange(0, 10, 2)
e = np.linspace(0, 1, 5)
f = np.eye(3)
g = np.random.randn(3, 3)

print(f"1D array: shape={a.shape}, dtype={a.dtype}")
print(f"Zeros: shape={b.shape}")
print(f"3D ones: shape={c.shape}")
print(f"Arange: {d}")
print(f"Linspace: {e}")
print(f"Identity:\n{f}")
print(f"Random:\n{g}")

In [ ]:
# Reshaping
x = np.arange(12)
print(f"Original: {x.shape} -> {x}")
print(f"Reshaped (3,4):\n{x.reshape(3,4)}")
print(f"Reshaped (2,2,3):\n{x.reshape(2,2,3)}")
print(f"Flatten: {x.reshape(3,4).ravel()}")
print(f"\nUsing -1 for auto-dimension: {x.reshape(3,-1).shape}")
print(f"Adding dimension: {x.shape} -> {x[np.newaxis,:].shape} (newaxis)")
print(f"Adding dimension: {x.shape} -> {x[:,np.newaxis].shape} (newaxis)")

## 2. Broadcasting Rules

In [ ]:
# Broadcasting: how NumPy handles arrays of different shapes
# Rule 1: If arrays differ in ndim, prepend 1s to the smaller shape
# Rule 2: Arrays with size 1 along a dimension act as if they had the largest size

print("=== Broadcasting Examples ===\n")

# (3,) + (3,) -> element-wise
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])
print(f"(3,) + (3,) = {a + b}  [element-wise]")

# (3,1) + (1,4) -> (3,4)
a = np.arange(3).reshape(3,1)
b = np.arange(4).reshape(1,4)
print(f"\n(3,1) + (1,4) -> shape {(a+b).shape}:")
print(a + b)

# scalar + array
print(f"\nscalar + (3,): {5 + np.array([1,2,3])}")

# Visual representation
print("\n=== Visual Broadcasting ===")
print("A (3,1):    B (1,4):    A+B (3,4):")
print("[[0]        [[0 1 2 3]   [[0 1 2 3]")
print(" [1]    +              =   [1 2 3 4]")
print(" [2]]                      [2 3 4 5]]")

## 3. Vectorization Speed Comparison

In [ ]:
# Loop vs vectorized operations
sizes = [100, 1000, 10000, 100000, 1000000]
loop_times = []
vec_times = []

for n in sizes:
    a = np.random.randn(n)
    b = np.random.randn(n)
    
    # Loop version
    start = time.perf_counter()
    result = np.empty(n)
    for i in range(n):
        result[i] = a[i] * b[i] + a[i]**2
    loop_times.append(time.perf_counter() - start)
    
    # Vectorized
    start = time.perf_counter()
    result = a * b + a**2
    vec_times.append(time.perf_counter() - start)

plt.figure(figsize=(10, 5))
plt.loglog(sizes, loop_times, 'ro-', label='Python Loop', lw=2)
plt.loglog(sizes, vec_times, 'bs-', label='NumPy Vectorized', lw=2)
plt.xlabel('Array Size'); plt.ylabel('Time (seconds)')
plt.title('Vectorization Speedup')
plt.legend()
plt.grid(True, alpha=0.3)

# Print speedups
for s, lt, vt in zip(sizes, loop_times, vec_times):
    print(f"Size {s:>8}: Loop={lt:.4f}s, Vec={vt:.6f}s, Speedup={lt/vt:.0f}x")
plt.show()

## 4. Matrix Operations for ML

In [ ]:
# Essential matrix operations
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("A:\n", A)
print("\nB:\n", B)
print("\nDot product (A @ B):\n", A @ B)
print("\nTranspose (A.T):\n", A.T)
print("\nInverse (np.linalg.inv(A)):\n", np.linalg.inv(A))
print("\nDeterminant:", np.linalg.det(A))
print("\nEigenvalues:", np.linalg.eig(A)[0])

# Verify inverse
print("\nA @ A_inv (should be identity):")
print(np.round(A @ np.linalg.inv(A), 10))

In [ ]:
# Practical: SVD for dimensionality reduction
np.random.seed(42)
# Create correlated data
data = np.random.randn(100, 2) @ np.array([[2, 1], [1, 3]])

U, S, Vt = np.linalg.svd(data, full_matrices=False)
print(f"Singular values: {S}")
print(f"Explained variance ratio: {S**2 / np.sum(S**2)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(data[:,0], data[:,1], alpha=0.5)
# Plot principal components
for i in range(2):
    axes[0].arrow(0, 0, Vt[i,0]*S[i]*0.3, Vt[i,1]*S[i]*0.3, 
                  head_width=0.3, color=['red','blue'][i], lw=2)
axes[0].set_title('Original Data with SVD Directions')
axes[0].set_aspect('equal')

# Project to 1D
projected = data @ Vt[0]
axes[1].hist(projected, bins=20, edgecolor='black')
axes[1].set_title('Projected to First Principal Component')
plt.tight_layout()
plt.show()

## 5. Linear Regression with Pure NumPy

In [ ]:
# Generate data
np.random.seed(42)
n_samples = 100
X = 2 * np.random.rand(n_samples, 1)
y = 4 + 3 * X.squeeze() + np.random.randn(n_samples) * 0.5

# Add bias term
X_b = np.c_[np.ones((n_samples, 1)), X]

# Normal equation: theta = (X^T X)^-1 X^T y
theta_analytical = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
print(f"Analytical solution: intercept={theta_analytical[0]:.4f}, slope={theta_analytical[1]:.4f}")
print(f"True values: intercept=4.0, slope=3.0")

# Gradient descent version
theta_gd = np.random.randn(2)
lr = 0.1
losses = []
for i in range(100):
    predictions = X_b @ theta_gd
    error = predictions - y
    loss = np.mean(error**2)
    losses.append(loss)
    gradient = 2/n_samples * X_b.T @ error
    theta_gd -= lr * gradient

print(f"GD solution: intercept={theta_gd[0]:.4f}, slope={theta_gd[1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X, y, alpha=0.5)
x_line = np.linspace(0, 2, 100)
axes[0].plot(x_line, theta_analytical[0] + theta_analytical[1]*x_line, 'r-', lw=2, label='Fit')
axes[0].set_title('Linear Regression'); axes[0].legend()
axes[1].plot(losses, 'b-')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('MSE')
axes[1].set_title('GD Convergence')
plt.tight_layout()
plt.show()

## 6. Random Number Generation and Sampling

In [ ]:
np.random.seed(42)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
distributions = [
    ('Normal(0,1)', np.random.randn(10000)),
    ('Uniform(0,1)', np.random.rand(10000)),
    ('Exponential(1)', np.random.exponential(1, 10000)),
    ('Binomial(10,0.5)', np.random.binomial(10, 0.5, 10000)),
    ('Poisson(5)', np.random.poisson(5, 10000)),
    ('Beta(2,5)', np.random.beta(2, 5, 10000)),
]

for ax, (name, samples) in zip(axes.flat, distributions):
    ax.hist(samples, bins=50, density=True, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.set_title(name)
    ax.set_ylabel('Density')

plt.suptitle('Common Probability Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Memory Layout and Performance Tips

In [ ]:
# C-order (row-major) vs F-order (column-major)
a_c = np.zeros((1000, 1000), order='C')
a_f = np.zeros((1000, 1000), order='F')

# Row access (fast in C-order)
start = time.perf_counter()
for i in range(1000):
    _ = a_c[i, :].sum()
c_row = time.perf_counter() - start

start = time.perf_counter()
for i in range(1000):
    _ = a_f[i, :].sum()
f_row = time.perf_counter() - start

# Column access (fast in F-order)
start = time.perf_counter()
for i in range(1000):
    _ = a_c[:, i].sum()
c_col = time.perf_counter() - start

start = time.perf_counter()
for i in range(1000):
    _ = a_f[:, i].sum()
f_col = time.perf_counter() - start

print("Memory Layout Performance (lower is better):")
print(f"{'':20} {'C-order':>10} {'F-order':>10}")
print(f"{'Row access':20} {c_row*1000:>8.2f}ms {f_row*1000:>8.2f}ms")
print(f"{'Column access':20} {c_col*1000:>8.2f}ms {f_col*1000:>8.2f}ms")
print(f"\nKey insight: Access data along contiguous memory for best performance!")
print(f"C-order: rows contiguous | F-order: columns contiguous")

## 8. Advanced Indexing

In [ ]:
# Boolean masking
data = np.random.randn(5, 4)
print("Data:\n", np.round(data, 2))
print("\nPositive values only:", data[data > 0])
print("\nRows where first column > 0:")
print(data[data[:, 0] > 0])

In [ ]:
# Fancy indexing
arr = np.arange(20).reshape(4, 5)
print("Array:\n", arr)
print("\nFancy index rows [0,2,3]:\n", arr[[0, 2, 3]])
print("\nFancy index specific elements arr[[0,1,2], [1,3,4]]:", arr[[0,1,2], [1,3,4]])

# Practical: top-k selection
scores = np.random.randn(10)
top_3_idx = np.argsort(scores)[-3:][::-1]
print(f"\nScores: {np.round(scores, 2)}")
print(f"Top 3 indices: {top_3_idx}")
print(f"Top 3 values: {np.round(scores[top_3_idx], 2)}")

In [ ]:
# np.where - conditional selection
x = np.random.randn(10)
result = np.where(x > 0, x, 0)  # ReLU!
print(f"Input:  {np.round(x, 2)}")
print(f"ReLU:   {np.round(result, 2)}")

# Practical: implementing softmax
def softmax(x):
    e_x = np.exp(x - np.max(x))  # subtract max for numerical stability
    return e_x / e_x.sum()

logits = np.array([2.0, 1.0, 0.1])
print(f"\nLogits: {logits}")
print(f"Softmax: {np.round(softmax(logits), 4)}")
print(f"Sum: {softmax(logits).sum()}")

## Key Tips

1. **Always vectorize** - avoid Python loops over array elements
2. **Use broadcasting** instead of explicit tiling/repeating
3. **Preallocate arrays** when you must use loops
4. **Use `@` for matrix multiply** (cleaner than `np.dot`)
5. **Subtract max before exp** for numerical stability